# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Before writing the rule, I check the two signals it leans on are real. Feature window: March 2026 (month='2026-03')
what the rule is allowed to see.
Outcome window: April 2026 (month='2026-04') used only to check whether the signals predict what actually happened next, never fed into the rule itself.

In [7]:
import os, getpass
import duckdb
import pandas as pd

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

FEATURE_MONTH = '2026-03'   # what the rule is allowed to see
OUTCOME_MONTH = '2026-04'   # only used to CHECK the signal — never fed into the rule

con.execute(f"CREATE OR REPLACE TEMPORARY VIEW dim_content_view AS SELECT * FROM {DIM_CONTENT}")
con.sql("DESCRIBE dim_content_view").df()

Paste your Hugging Face READ token (hf_...): ··········


,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


In [15]:
monthly = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
        SUM(CASE WHEN month = '{FEATURE_MONTH}' THEN gsc_impressions ELSE 0 END) AS imp_march,
        SUM(CASE WHEN month = '{FEATURE_MONTH}' THEN gsc_clicks ELSE 0 END) AS clk_march,
        AVG(CASE WHEN month = '{FEATURE_MONTH}' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS pos_march,
        SUM(CASE WHEN month = '{OUTCOME_MONTH}' THEN gsc_impressions ELSE 0 END) AS imp_april
    FROM {FACT}
    WHERE month IN ('{FEATURE_MONTH}', '{OUTCOME_MONTH}')
    GROUP BY 1,2
    HAVING imp_march >= 100
""").df()

monthly['ctr_march'] = (monthly['clk_march'] / monthly['imp_march']).round(4)
monthly['declined_next_month'] = (monthly['imp_april'] < 0.8 * monthly['imp_march']).astype(int)

content = con.sql(f"""
    SELECT
        content_hash_id,
        content_updated_date,
        DATE_DIFF('day', content_updated_date, DATE '{FEATURE_MONTH}-01') AS days_since_last_update
    FROM {DIM_CONTENT}
""").df()

df = monthly.merge(content, on='content_hash_id', how='left')

# dim_content is a single current-state snapshot, not time-versioned. If a page's
# update date falls AFTER our March window, we can't honestly claim its March
# staleness — mark it unknown rather than silently treating it as "not stale."
df.loc[df['days_since_last_update'] < 0, 'days_since_last_update'] = pd.NA
n_unknown = df['days_since_last_update'].isna().sum()
print(f"{len(df):,} total rows; {n_unknown:,} have unknown March staleness "
      "(updated after March, or never recorded) — excluded from Signal 1 below.")
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

101,441 total rows; 83,426 have unknown March staleness (updated after March, or never recorded) — excluded from Signal 1 below.


,client_hash_id,content_hash_id,imp_march,clk_march,pos_march,imp_april,ctr_march,declined_next_month,content_updated_date,days_since_last_update
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,2.0,4.394234,1151.0,0.0018,0,2026-05-18,NaN
1,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,0.0,8.454069,98.0,0.0000,1,2026-05-18,NaN
2,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.0,6.320337,2275.0,0.0042,0,2026-05-18,NaN
3,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,16.0,4.459107,6266.0,0.0058,0,2026-07-06,NaN
4,client_73cda7b4e4f265ea,content_2662845f598544ef,150.0,1.0,7.046534,100.0,0.0067,1,2026-05-18,NaN


In [16]:
known_staleness = df.dropna(subset=['days_since_last_update'])
known_staleness = known_staleness.copy()
known_staleness['staleness_bucket'] = pd.cut(
    known_staleness['days_since_last_update'],
    bins=[-1, 90, 180, 365, 100000],
    labels=['<90d', '90-180d', '180-365d', '365d+']
)
signal1 = known_staleness.groupby('staleness_bucket', observed=True).agg(
    n=('declined_next_month', 'size'),
    decline_rate=('declined_next_month', 'mean')
).round(3)
signal1

,n,decline_rate
staleness_bucket,,
<90d,17898,0.608
90-180d,99,0.525
180-365d,18,0.944


Signal 1 (staleness) verdict:
MIXED. decline_rate goes 0.608 (<90d, n=17,898) → 0.525 (90-180d, n=99) → 0.944 (180-365d, n=18) not a clean monotonic rise, and the two buckets that matter most for the rule have very small n (99 and 18). Directionally the most-stale bucket does have the highest decline rate. It also means only a small, determinable slice of the panel (18,015 of 101,441 rows) could be checked at all most content's update date fell after March and had to be excluded.

In [17]:
visible = df[(df['imp_march'] >= 500) & (df['pos_march'] > 0) & (df['pos_march'] <= 20)].copy()
visible['ctr_bucket'] = pd.cut(
    visible['ctr_march'], bins=[-1, 0.005, 0.02, 0.05, 1],
    labels=['<0.5%', '0.5-2%', '2-5%', '5%+']
)
signal2 = visible.groupby('ctr_bucket', observed=True).agg(
    n=('declined_next_month', 'size'),
    decline_rate=('declined_next_month', 'mean')
).round(3)
signal2

,n,decline_rate
ctr_bucket,,
<0.5%,41273,0.542
0.5-2%,9141,0.295
2-5%,292,0.144
5%+,11,0.455


Signal 2 (CTR-vs-position) verdict:

CONFIRMED - decline_rate falls steadily as CTR rises across the three well-populated buckets (0.542 → 0.295 → 0.144), n=50,706 in those three combined. The 5%+ bucket (n=11) bounces back up but is too small to trust noise, not a real reversal.

A page is worth reviewing for refresh if it's stale (no update in 180+ days, as far as we can determine) AND it still gets meaningful search visibility (500+ impressions in March).

Reason code: stale_visible_page.

Note: for 82% of the panel, the update date couldn't be verified as of March, so those pages are conservatively scored as "not stale" rather than guessed this makes the rule cautious, not aggressive.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [19]:
df['stale'] = (df['days_since_last_update'] >= 180).fillna(False).astype(int)
df['visible'] = (df['imp_march'] >= 500).astype(int)
df['score'] = df['stale'] * df['visible'] * df['imp_march']
df['reason_code'] = 'stale_visible_page'
df['action'] = df['score'].apply(lambda s: 'review_for_refresh' if s > 0 else 'monitor')

queue = df.sort_values('score', ascending=False)[
    ['client_hash_id', 'content_hash_id', 'imp_march', 'days_since_last_update', 'score', 'reason_code', 'action']
]

os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)
print(f"Wrote {len(queue):,} rows to work/outputs/baseline_action_score.csv")
queue.head(10)

Wrote 101,441 rows to work/outputs/baseline_action_score.csv


,client_hash_id,content_hash_id,imp_march,days_since_last_update,score,reason_code,action
4319,client_c182d11e4862a37d,content_42ce26be1ec6be00,4411.0,234.0,4411.0,stale_visible_page,review_for_refresh
12314,client_c182d11e4862a37d,content_bea86ce3455100b0,3670.0,202.0,3670.0,stale_visible_page,review_for_refresh
4335,client_c182d11e4862a37d,content_5120dcbbb086843d,1429.0,217.0,1429.0,stale_visible_page,review_for_refresh
54812,client_65de48885f4ef01b,content_eba53d72e18a9f93,734.0,201.0,734.0,stale_visible_page,review_for_refresh
4004,client_65de48885f4ef01b,content_c126a43258b574c3,592.0,201.0,592.0,stale_visible_page,review_for_refresh
63084,client_c182d11e4862a37d,content_5271624ae98fff86,550.0,201.0,550.0,stale_visible_page,review_for_refresh
67615,client_23a62021009f63c4,content_453abf7830acb4b4,8214.0,NaN,0.0,stale_visible_page,monitor
67616,client_23a62021009f63c4,content_112aee06ac44905a,9889.0,NaN,0.0,stale_visible_page,monitor
67635,client_23a62021009f63c4,content_9f64243a4d5963f9,5378.0,NaN,0.0,stale_visible_page,monitor
67634,client_23a62021009f63c4,content_2fe3d9409fda254d,1263.0,4.0,0.0,stale_visible_page,monitor


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [20]:
top20 = queue.head(20).merge(
    df[['content_hash_id', 'ctr_march', 'pos_march', 'declined_next_month']],
    on='content_hash_id'
)
top20

,client_hash_id,content_hash_id,imp_march,days_since_last_update,score,reason_code,action,ctr_march,pos_march,declined_next_month
0,client_c182d11e4862a37d,content_42ce26be1ec6be00,4411.0,234.0,4411.0,stale_visible_page,review_for_refresh,0.0014,4.262553,1
1,client_c182d11e4862a37d,content_bea86ce3455100b0,3670.0,202.0,3670.0,stale_visible_page,review_for_refresh,0.0003,6.555793,1
2,client_c182d11e4862a37d,content_5120dcbbb086843d,1429.0,217.0,1429.0,stale_visible_page,review_for_refresh,0.0000,6.321173,1
3,client_65de48885f4ef01b,content_eba53d72e18a9f93,734.0,201.0,734.0,stale_visible_page,review_for_refresh,0.0027,5.234187,1
4,client_65de48885f4ef01b,content_c126a43258b574c3,592.0,201.0,592.0,stale_visible_page,review_for_refresh,0.0000,29.522907,1
5,client_c182d11e4862a37d,content_5271624ae98fff86,550.0,201.0,550.0,stale_visible_page,review_for_refresh,0.0055,6.532108,1
6,client_23a62021009f63c4,content_453abf7830acb4b4,8214.0,NaN,0.0,stale_visible_page,monitor,0.0043,9.932932,1
7,client_23a62021009f63c4,content_112aee06ac44905a,9889.0,NaN,0.0,stale_visible_page,monitor,0.0009,15.891698,1
8,client_23a62021009f63c4,content_9f64243a4d5963f9,5378.0,NaN,0.0,stale_visible_page,monitor,0.0006,30.625812,1
9,client_23a62021009f63c4,content_2fe3d9409fda254d,1263.0,4.0,0.0,stale_visible_page,monitor,0.0008,20.134791,0


1. content_42ce26be1ec6be00 action: review_for_refresh; why: 234 days since last update, 4,411 March impressions; what would make it wrong: if this page is an intentionally-frozen reference page.

2. content_bea86ce3455100b0 action: review_for_refresh; why: 202 days stale, 3,670 impressions; what would make it wrong: same as above check if it's evergreen by design.

3. content_5120dcbbb086843d action: review_for_refresh; why: 217 days stale, 1,429 impressions; what would make it wrong: if position is already strong and the real issue is elsewhere.

4. content_eba53d72e18a9f93 (client_65de48885f4ef01b) action: review_for_refresh; why: 201 days stale, 734 impressions; what would make it wrong: impressions are modest may not be worth reviewer time vs. higher-volume pages.

5. content_c126a43258b574c3 action: review_for_refresh; why: 201 days stale, 592 impressions; what would make it wrong: same modest-volume concern as .

6. content_5271624ae98fff86 action: review_for_refresh; why: 201 days stale, 550 impressions; sits right at the visibility threshold what would make it wrong: borderline cases like this are the most threshold-sensitive; a slightly different cutoff excludes it entirely.

The remaining 14 rows in this slot all have score = 0.0 and action = monitor they are not real recommendations. The rule only produced 6 genuine review_for_refresh candidates across all 101,441 rows in this slice, almost entirely because 82% of the panel had no verifiable pre-March update date. .head(20) filled the remaining slots with zero-score ties rather than leaving them empty. This is itself the most important finding of this review: the rule's honest yield here is 6, not 20.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [26]:
print("Leakage check:")
print("- Rule inputs used:", ['content_updated_date (days_since_last_update)', 'imp_march'])
print("- April data touched ONLY to verify the signals — never fed into the score or rule.")
print("- No product flags (health_score, priority_score, action_type) used anywhere.")

weak = top20[top20['pos_march'] < 5]
weak

Leakage check:
- Rule inputs used: ['content_updated_date (days_since_last_update)', 'imp_march']
- April data touched ONLY to verify the signals — never fed into the score or rule.
- No product flags (health_score, priority_score, action_type) used anywhere.


,client_hash_id,content_hash_id,imp_march,days_since_last_update,score,reason_code,action,ctr_march,pos_march,declined_next_month
0,client_c182d11e4862a37d,content_42ce26be1ec6be00,4411.0,234.0,4411.0,stale_visible_page,review_for_refresh,0.0014,4.262553,1
6,client_e547b89c05043229,content_cf2264753938463b,46938.0,NaN,0.0,stale_visible_page,monitor,0.0002,4.111066,0
7,client_e547b89c05043229,content_90a17676972446b8,1255.0,NaN,0.0,stale_visible_page,monitor,0.0024,2.419881,0
8,client_e547b89c05043229,content_4220cf72969b6723,746.0,NaN,0.0,stale_visible_page,monitor,0.0080,4.093265,0
10,client_e547b89c05043229,content_01faaf022ac7e19b,3997.0,NaN,0.0,stale_visible_page,monitor,0.0025,3.922826,0
11,client_e547b89c05043229,content_6430f2d51284ca8f,1171.0,NaN,0.0,stale_visible_page,monitor,0.0085,3.719870,0
13,client_e547b89c05043229,content_9f008ee8d41d3aed,2376.0,NaN,0.0,stale_visible_page,monitor,0.0025,2.145977,0
14,client_e547b89c05043229,content_4cbf83e289d59e4a,2644.0,NaN,0.0,stale_visible_page,monitor,0.0049,3.224191,0
19,client_e547b89c05043229,content_02486d4dff97595d,3832.0,NaN,0.0,stale_visible_page,monitor,0.0044,3.906998,0


Weak pick: row 1 (content_42ce26be1ec6be00) is already ranking around position 4 in March with a very low CTR (0.14%) yet still declined into April. Flagging it as "review_for_refresh" may point the reviewer at the wrong fix a page ranking that well with that low a CTR looks more like a CTR/snippet problem (matches the low_ctr_visible_page pattern from Signal 2) than a staleness problem. The bigger weak point in this list overall is structural, not any one row: 14 of the "top 20" aren't real picks at all the rule's true candidate pool for this slice is just 6 pages, which is itself worth flagging to whoever consumes this queue.

## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.